## Homework 3: Symbolic Music Generation Using Markov Chains

**Before starting the homework:**

Please run `pip install miditok` to install the [MiDiTok](https://github.com/Natooz/MidiTok) package, which simplifies MIDI file processing by making note and beat extraction more straightforward.

You’re also welcome to experiment with other MIDI processing libraries such as [mido](https://github.com/mido/mido), [pretty_midi](https://github.com/craffel/pretty-midi) and [miditoolkit](https://github.com/YatingMusic/miditoolkit). However, with these libraries, you’ll need to handle MIDI quantization yourself, for example, converting note-on/note-off events into beat positions and durations.

In [ ]:
# run this command to install MiDiTok
#! pip install miditok

In [ ]:
# import required packages
import random
from glob import glob
from collections import defaultdict

import numpy as np
from numpy.random import choice

from symusic import Score
from miditok import REMI, TokenizerConfig
from midiutil import MIDIFile

In [ ]:
# You can change the random seed but try to keep your results deterministic!
# If I need to make changes to the autograder it'll require rerunning your code,
# so it should ideally generate the same results each time.
random.seed(42)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Load music dataset
We will use a subset of the [PDMX dataset](https://zenodo.org/records/14984509).

Please find the file `PDMX_subset.zip` in the homework spec.

All pieces are monophonic music (i.e. one melody line) in 4/4 time signature.

In [ ]:
midi_files = glob('PDMX_subset/*.mid')
len(midi_files)

### Train a tokenizer with the REMI method in MidiTok

In [ ]:
config = TokenizerConfig(num_velocities=1, use_chords=False, use_programs=False)
tokenizer = REMI(config)
tokenizer.train(vocab_size=1000, files_paths=midi_files)

### Use the trained tokenizer to get tokens for each midi file
In REMI representation, each note will be represented with four tokens: `Position, Pitch, Velocity, Duration`, e.g. `('Position_28', 'Pitch_74', 'Velocity_127', 'Duration_0.4.8')`; a `Bar_None` token indicates the beginning of a new bar.

In [ ]:
# e.g.:
midi = Score(midi_files[0])
tokens = tokenizer(midi)[0].tokens
tokens[:10]

1. Write a function to extract note pitch events from a midi file; and another extract all note pitch events from the dataset and output a dictionary that maps note pitch events to the number of times they occur in the files. (e.g. {60: 120, 61: 58, …}).

`note_extraction()`
- **Input**: a midi file

- **Output**: a list of note pitch events (e.g. [60, 62, 61, ...])

`note_frequency()`
- **Input**: all midi files `midi_files`

- **Output**: a dictionary that maps note pitch events to the number of times they occur, e.g {60: 120, 61: 58, …}

In [ ]:
_NOTE_CACHE = {}
_BEAT_CACHE = {}
_TRANSITION_CACHE = {}


def _tokens_for_file(midi_file):
    midi = Score(midi_file)
    return tokenizer(midi)[0].tokens


def _token_value(token):
    return token.split('_', 1)[1]


def _duration_to_length(duration):
    if duration in duration2length:
        return duration2length[duration]
    parts = duration.split('.')
    if len(parts) >= 2:
        return int(parts[0]) * 8 + int(parts[1])
    return int(duration)


def _normalize_counter(counter):
    total = sum(counter.values())
    if total == 0:
        return {}
    return {key: counter[key] / total for key in sorted(counter)}


def _probability_lookup(transitions, probabilities, key, value):
    if key not in transitions:
        return None
    try:
        index = transitions[key].index(value)
    except ValueError:
        return None
    return probabilities[key][index]


def _sample_from_distribution(values, probabilities):
    if len(values) == 1:
        return values[0]
    return int(choice(values, p=probabilities))


def _files_key(files):
    return tuple(files)


def _beat_unigram_probability(files):
    counts = defaultdict(int)
    for file in files:
        for _position, length in beat_extraction(file):
            counts[length] += 1
    return _normalize_counter(counts)


def note_extraction(midi_file):
    if midi_file in _NOTE_CACHE:
        return list(_NOTE_CACHE[midi_file])

    notes = []
    for token in _tokens_for_file(midi_file):
        if token.startswith('Pitch_'):
            notes.append(int(_token_value(token)))

    _NOTE_CACHE[midi_file] = tuple(notes)
    return notes

In [ ]:
def note_frequency(midi_files):
    note_counts = defaultdict(int)
    for midi_file in midi_files:
        for note in note_extraction(midi_file):
            note_counts[note] += 1
    return dict(sorted(note_counts.items()))

2. Write a function to normalize the above dictionary to produce probability scores (e.g. {60: 0.13, 61: 0.065, …})

`note_unigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: a dictionary that maps note pitch events to probabilities, e.g. {60: 0.13, 61: 0.06, …}

In [ ]:
def note_unigram_probability(midi_files):
    note_counts = note_frequency(midi_files)
    unigramProbabilities = {}

    total = sum(note_counts.values())
    if total > 0:
        for note in sorted(note_counts):
            unigramProbabilities[note] = note_counts[note] / total

    return unigramProbabilities

3. Generate a table of pairwise probabilities containing p(next_note | previous_note) values for the dataset; write a function that randomly generates the next note based on the previous note based on this distribution.

`note_bigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `bigramTransitions`: key: previous_note, value: a list of next_note, e.g. {60:[62, 64, ..], 62:[60, 64, ..], ...} (i.e., this is a list of every other note that occured after note 60, every note that occured after note 62, etc.)

  - `bigramTransitionProbabilities`: key:previous_note, value: a list of probabilities for next_note in the same order of `bigramTransitions`, e.g. {60:[0.3, 0.4, ..], 62:[0.2, 0.1, ..], ...} (i.e., you are converting the values above to probabilities)

`sample_next_note()`
- **Input**: a note

- **Output**: next note sampled from pairwise probabilities

In [ ]:
def note_bigram_probability(midi_files):
    cache_key = ('note_bigram', _files_key(midi_files))
    if cache_key in _TRANSITION_CACHE:
        return _TRANSITION_CACHE[cache_key]

    transition_counts = defaultdict(lambda: defaultdict(int))
    for midi_file in midi_files:
        notes = note_extraction(midi_file)
        for previous_note, next_note in zip(notes[:-1], notes[1:]):
            transition_counts[previous_note][next_note] += 1

    bigramTransitions = defaultdict(list)
    bigramTransitionProbabilities = defaultdict(list)
    for previous_note in sorted(transition_counts):
        next_counts = transition_counts[previous_note]
        total = sum(next_counts.values())
        for next_note in sorted(next_counts):
            bigramTransitions[previous_note].append(next_note)
            bigramTransitionProbabilities[previous_note].append(next_counts[next_note] / total)

    _TRANSITION_CACHE[cache_key] = (bigramTransitions, bigramTransitionProbabilities)
    return bigramTransitions, bigramTransitionProbabilities

In [ ]:
def sample_next_note(note):
    bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)
    if note in bigramTransitions and len(bigramTransitions[note]) > 0:
        return _sample_from_distribution(bigramTransitions[note], bigramTransitionProbabilities[note])

    unigramProbabilities = note_unigram_probability(midi_files)
    return _sample_from_distribution(list(unigramProbabilities.keys()), list(unigramProbabilities.values()))

4. Write a function to calculate the perplexity of your model on a midi file.

    The perplexity of a model is defined as

    $\quad \text{exp}(-\frac{1}{N} \sum_{i=1}^N \text{log}(p(w_i|w_{i-1})))$

    where $p(w_1|w_0) = p(w_1)$, $p(w_i|w_{i-1}) (i>1)$ refers to the pairwise probability p(next_note | previous_note).

`note_bigram_perplexity()`
- **Input**: a midi file

- **Output**: perplexity value

In [ ]:
def note_bigram_perplexity(midi_file):
    unigramProbabilities = note_unigram_probability(midi_files)
    bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)

    notes = note_extraction(midi_file)
    if len(notes) == 0:
        return float('inf')

    eps = 1e-12
    log_probability = np.log(max(unigramProbabilities.get(notes[0], eps), eps))
    for previous_note, current_note in zip(notes[:-1], notes[1:]):
        probability = _probability_lookup(bigramTransitions, bigramTransitionProbabilities, previous_note, current_note)
        if probability is None:
            probability = unigramProbabilities.get(current_note, eps)
        log_probability += np.log(max(probability, eps))

    return float(np.exp(-log_probability / len(notes)))

5. Implement a second-order Markov chain, i.e., one which estimates p(next_note | next_previous_note, previous_note); write a function to compute the perplexity of this new model on a midi file.

    The perplexity of this model is defined as

    $\quad \text{exp}(-\frac{1}{N} \sum_{i=1}^N \text{log}(p(w_i|w_{i-2}, w_{i-1})))$

    where $p(w_1|w_{-1}, w_0) = p(w_1)$, $p(w_2|w_0, w_1) = p(w_2|w_1)$, $p(w_i|w_{i-2}, w_{i-1}) (i>2)$ refers to the probability p(next_note | next_previous_note, previous_note).


`note_trigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `trigramTransitions`: key - (next_previous_note, previous_note), value - a list of next_note, e.g. {(60, 62):[64, 66, ..], (60, 64):[60, 64, ..], ...}

  - `trigramTransitionProbabilities`: key: (next_previous_note, previous_note), value: a list of probabilities for next_note in the same order of `trigramTransitions`, e.g. {(60, 62):[0.2, 0.2, ..], (60, 64):[0.4, 0.1, ..], ...}

`note_trigram_perplexity()`
- **Input**: a midi file

- **Output**: perplexity value

In [ ]:
def note_trigram_probability(midi_files):
    cache_key = ('note_trigram', _files_key(midi_files))
    if cache_key in _TRANSITION_CACHE:
        return _TRANSITION_CACHE[cache_key]

    transition_counts = defaultdict(lambda: defaultdict(int))
    for midi_file in midi_files:
        notes = note_extraction(midi_file)
        for index in range(2, len(notes)):
            key = (notes[index - 2], notes[index - 1])
            transition_counts[key][notes[index]] += 1

    trigramTransitions = defaultdict(list)
    trigramTransitionProbabilities = defaultdict(list)
    for key in sorted(transition_counts):
        next_counts = transition_counts[key]
        total = sum(next_counts.values())
        for next_note in sorted(next_counts):
            trigramTransitions[key].append(next_note)
            trigramTransitionProbabilities[key].append(next_counts[next_note] / total)

    _TRANSITION_CACHE[cache_key] = (trigramTransitions, trigramTransitionProbabilities)
    return trigramTransitions, trigramTransitionProbabilities

In [ ]:
def note_trigram_perplexity(midi_file):
    unigramProbabilities = note_unigram_probability(midi_files)
    bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)
    trigramTransitions, trigramTransitionProbabilities = note_trigram_probability(midi_files)

    notes = note_extraction(midi_file)
    if len(notes) == 0:
        return float('inf')

    eps = 1e-12
    log_probability = np.log(max(unigramProbabilities.get(notes[0], eps), eps))

    if len(notes) >= 2:
        probability = _probability_lookup(bigramTransitions, bigramTransitionProbabilities, notes[0], notes[1])
        if probability is None:
            probability = unigramProbabilities.get(notes[1], eps)
        log_probability += np.log(max(probability, eps))

    for index in range(2, len(notes)):
        key = (notes[index - 2], notes[index - 1])
        probability = _probability_lookup(trigramTransitions, trigramTransitionProbabilities, key, notes[index])
        if probability is None:
            probability = _probability_lookup(bigramTransitions, bigramTransitionProbabilities, notes[index - 1], notes[index])
        if probability is None:
            probability = unigramProbabilities.get(notes[index], eps)
        log_probability += np.log(max(probability, eps))

    return float(np.exp(-log_probability / len(notes)))

6. Our model currently doesn’t have any knowledge of beats. Write a function that extracts beat lengths and outputs a list of [(beat position; beat length)] values.

    Recall that each note will be encoded as `Position, Pitch, Velocity, Duration` using REMI. Please keep the `Position` value for beat position, and convert `Duration` to beat length using provided lookup table `duration2length` (see below).

    For example, for a note represented by four tokens `('Position_24', 'Pitch_72', 'Velocity_127', 'Duration_0.4.8')`, the extracted (beat position; beat length) value is `(24, 4)`.

    As a result, we will obtain a list like [(0,8),(8,16),(24,4),(28,4),(0,4)...], where the next beat position is the previous beat position + the beat length. As we divide each bar into 32 positions by default, when reaching the end of a bar (i.e. 28 + 4 = 32 in the case of (28, 4)), the beat position reset to 0.

In [ ]:
duration2length = {
    '0.2.8': 2,  # sixteenth note, 0.25 beat in 4/4 time signature
    '0.4.8': 4,  # eighth note, 0.5 beat in 4/4 time signature
    '1.0.8': 8,  # quarter note, 1 beat in 4/4 time signature
    '2.0.8': 16, # half note, 2 beats in 4/4 time signature
    '4.0.4': 32, # whole note, 4 beats in 4/4 time signature
}

`beat_extraction()`
- **Input**: a midi file

- **Output**: a list of (beat position; beat length) values

In [ ]:
def beat_extraction(midi_file):
    if midi_file in _BEAT_CACHE:
        return list(_BEAT_CACHE[midi_file])

    beats = []
    current_position = None
    for token in _tokens_for_file(midi_file):
        if token.startswith('Position_'):
            current_position = int(_token_value(token))
        elif token.startswith('Duration_') and current_position is not None:
            duration = _token_value(token)
            beats.append((current_position, _duration_to_length(duration)))

    _BEAT_CACHE[midi_file] = tuple(beats)
    return beats

7. Implement a Markov chain that computes p(beat_length | previous_beat_length) based on the above function.

`beat_bigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `bigramBeatTransitions`: key: previous_beat_length, value: a list of beat_length, e.g. {4:[8, 2, ..], 8:[8, 4, ..], ...}

  - `bigramBeatTransitionProbabilities`: key - previous_beat_length, value - a list of probabilities for beat_length in the same order of `bigramBeatTransitions`, e.g. {4:[0.3, 0.2, ..], 8:[0.4, 0.4, ..], ...}

In [ ]:
def beat_bigram_probability(midi_files):
    cache_key = ('beat_bigram', _files_key(midi_files))
    if cache_key in _TRANSITION_CACHE:
        return _TRANSITION_CACHE[cache_key]

    transition_counts = defaultdict(lambda: defaultdict(int))
    for midi_file in midi_files:
        beat_lengths = [length for _position, length in beat_extraction(midi_file)]
        for previous_length, current_length in zip(beat_lengths[:-1], beat_lengths[1:]):
            transition_counts[previous_length][current_length] += 1

    bigramBeatTransitions = defaultdict(list)
    bigramBeatTransitionProbabilities = defaultdict(list)
    for previous_length in sorted(transition_counts):
        next_counts = transition_counts[previous_length]
        total = sum(next_counts.values())
        for current_length in sorted(next_counts):
            bigramBeatTransitions[previous_length].append(current_length)
            bigramBeatTransitionProbabilities[previous_length].append(next_counts[current_length] / total)

    _TRANSITION_CACHE[cache_key] = (bigramBeatTransitions, bigramBeatTransitionProbabilities)
    return bigramBeatTransitions, bigramBeatTransitionProbabilities

8. Implement a function to compute p(beat length | beat position), and compute the perplexity of your models from Q7 and Q8. For both models, we only consider the probabilities of predicting the sequence of **beat lengths**.

`beat_pos_bigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `bigramBeatPosTransitions`: key - beat_position, value - a list of beat_length

  - `bigramBeatPosTransitionProbabilities`: key - beat_position, value - a list of probabilities for beat_length in the same order of `bigramBeatPosTransitions`

`beat_bigram_perplexity()`
- **Input**: a midi file

- **Output**: two perplexity values correspond to the models in Q7 and Q8, respectively

In [ ]:
def beat_pos_bigram_probability(midi_files):
    cache_key = ('beat_position_bigram', _files_key(midi_files))
    if cache_key in _TRANSITION_CACHE:
        return _TRANSITION_CACHE[cache_key]

    transition_counts = defaultdict(lambda: defaultdict(int))
    for midi_file in midi_files:
        for beat_position, beat_length in beat_extraction(midi_file):
            transition_counts[beat_position][beat_length] += 1

    bigramBeatPosTransitions = defaultdict(list)
    bigramBeatPosTransitionProbabilities = defaultdict(list)
    for beat_position in sorted(transition_counts):
        next_counts = transition_counts[beat_position]
        total = sum(next_counts.values())
        for beat_length in sorted(next_counts):
            bigramBeatPosTransitions[beat_position].append(beat_length)
            bigramBeatPosTransitionProbabilities[beat_position].append(next_counts[beat_length] / total)

    _TRANSITION_CACHE[cache_key] = (bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities)
    return bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities

In [ ]:
def beat_bigram_perplexity(midi_file):
    bigramBeatTransitions, bigramBeatTransitionProbabilities = beat_bigram_probability(midi_files)
    bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities = beat_pos_bigram_probability(midi_files)

    beats = beat_extraction(midi_file)
    if len(beats) == 0:
        return float('inf'), float('inf')

    eps = 1e-12
    beatUnigramProbabilities = _beat_unigram_probability(midi_files)
    beat_lengths = [length for _position, length in beats]

    log_probability_Q7 = np.log(max(beatUnigramProbabilities.get(beat_lengths[0], eps), eps))
    for previous_length, current_length in zip(beat_lengths[:-1], beat_lengths[1:]):
        probability = _probability_lookup(bigramBeatTransitions, bigramBeatTransitionProbabilities, previous_length, current_length)
        if probability is None:
            probability = beatUnigramProbabilities.get(current_length, eps)
        log_probability_Q7 += np.log(max(probability, eps))

    log_probability_Q8 = 0.0
    for beat_position, beat_length in beats:
        probability = _probability_lookup(bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities, beat_position, beat_length)
        if probability is None:
            probability = beatUnigramProbabilities.get(beat_length, eps)
        log_probability_Q8 += np.log(max(probability, eps))

    perplexity_Q7 = float(np.exp(-log_probability_Q7 / len(beat_lengths)))
    perplexity_Q8 = float(np.exp(-log_probability_Q8 / len(beat_lengths)))

    return perplexity_Q7, perplexity_Q8

9. Implement a Markov chain that computes p(beat_length | previous_beat_length, beat_position), and report its perplexity.

`beat_trigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `trigramBeatTransitions`: key: (previous_beat_length, beat_position), value: a list of beat_length

  - `trigramBeatTransitionProbabilities`: key: (previous_beat_length, beat_position), value: a list of probabilities for beat_length in the same order of `trigramBeatTransitions`

`beat_trigram_perplexity()`
- **Input**: a midi file

- **Output**: perplexity value

In [ ]:
def beat_trigram_probability(midi_files):
    cache_key = ('beat_trigram', _files_key(midi_files))
    if cache_key in _TRANSITION_CACHE:
        return _TRANSITION_CACHE[cache_key]

    transition_counts = defaultdict(lambda: defaultdict(int))
    for midi_file in midi_files:
        beats = beat_extraction(midi_file)
        for index in range(1, len(beats)):
            previous_length = beats[index - 1][1]
            beat_position, beat_length = beats[index]
            transition_counts[(previous_length, beat_position)][beat_length] += 1

    trigramBeatTransitions = defaultdict(list)
    trigramBeatTransitionProbabilities = defaultdict(list)
    for key in sorted(transition_counts):
        next_counts = transition_counts[key]
        total = sum(next_counts.values())
        for beat_length in sorted(next_counts):
            trigramBeatTransitions[key].append(beat_length)
            trigramBeatTransitionProbabilities[key].append(next_counts[beat_length] / total)

    _TRANSITION_CACHE[cache_key] = (trigramBeatTransitions, trigramBeatTransitionProbabilities)
    return trigramBeatTransitions, trigramBeatTransitionProbabilities

In [ ]:
def beat_trigram_perplexity(midi_file):
    bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities = beat_pos_bigram_probability(midi_files)
    trigramBeatTransitions, trigramBeatTransitionProbabilities = beat_trigram_probability(midi_files)

    beats = beat_extraction(midi_file)
    if len(beats) == 0:
        return float('inf')

    eps = 1e-12
    beatUnigramProbabilities = _beat_unigram_probability(midi_files)

    first_position, first_length = beats[0]
    probability = _probability_lookup(bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities, first_position, first_length)
    if probability is None:
        probability = beatUnigramProbabilities.get(first_length, eps)
    log_probability = np.log(max(probability, eps))

    for index in range(1, len(beats)):
        previous_length = beats[index - 1][1]
        beat_position, beat_length = beats[index]
        probability = _probability_lookup(trigramBeatTransitions, trigramBeatTransitionProbabilities, (previous_length, beat_position), beat_length)
        if probability is None:
            probability = _probability_lookup(bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities, beat_position, beat_length)
        if probability is None:
            probability = beatUnigramProbabilities.get(beat_length, eps)
        log_probability += np.log(max(probability, eps))

    return float(np.exp(-log_probability / len(beats)))

10. Use the model from Q5 to generate N notes, and the model from Q8 to generate beat lengths for each note. Save the generated music as a midi file (see code from workbook1) as q10.mid. Remember to reset the beat position to 0 when reaching the end of a bar.

`music_generate`
- **Input**: target length, e.g. 500

- **Output**: a midi file q10.mid

Note: the duration of one beat in MIDIUtil is 1, while in MidiTok is 8. Divide beat length by 8 if you use methods in MIDIUtil to save midi files.

In [ ]:
def music_generate(length):
    # sample notes
    unigramProbabilities = note_unigram_probability(midi_files)
    bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)
    trigramTransitions, trigramTransitionProbabilities = note_trigram_probability(midi_files)

    random.seed(42)
    sampled_notes = []
    if length > 0:
        sampled_notes.append(_sample_from_distribution(list(unigramProbabilities.keys()), list(unigramProbabilities.values())))
    if length > 1:
        sampled_notes.append(_sample_from_distribution(bigramTransitions[sampled_notes[0]], bigramTransitionProbabilities[sampled_notes[0]]))
    while len(sampled_notes) < length:
        key = (sampled_notes[-2], sampled_notes[-1])
        if key in trigramTransitions and len(trigramTransitions[key]) > 0:
            next_note = _sample_from_distribution(trigramTransitions[key], trigramTransitionProbabilities[key])
        elif sampled_notes[-1] in bigramTransitions and len(bigramTransitions[sampled_notes[-1]]) > 0:
            next_note = _sample_from_distribution(bigramTransitions[sampled_notes[-1]], bigramTransitionProbabilities[sampled_notes[-1]])
        else:
            next_note = _sample_from_distribution(list(unigramProbabilities.keys()), list(unigramProbabilities.values()))
        sampled_notes.append(next_note)

    # sample beats
    bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities = beat_pos_bigram_probability(midi_files)
    beatUnigramProbabilities = _beat_unigram_probability(midi_files)
    sampled_beats = []
    beat_position = 0
    for _ in range(length):
        if beat_position in bigramBeatPosTransitions and len(bigramBeatPosTransitions[beat_position]) > 0:
            beat_length = _sample_from_distribution(bigramBeatPosTransitions[beat_position], bigramBeatPosTransitionProbabilities[beat_position])
        else:
            beat_length = _sample_from_distribution(list(beatUnigramProbabilities.keys()), list(beatUnigramProbabilities.values()))
        sampled_beats.append((beat_position, beat_length))
        beat_position = (beat_position + beat_length) % 32

    # save the generated music as a midi file
    generated_midi = MIDIFile(1)
    track = 0
    channel = 0
    time = 0.0
    tempo = 120
    volume = 100
    generated_midi.addTempo(track, time, tempo)

    for note, (_beat_position, beat_length) in zip(sampled_notes, sampled_beats):
        duration = beat_length / 8.0
        generated_midi.addNote(track, channel, int(note), time, duration, volume)
        time += duration

    with open('q10.mid', 'wb') as output_file:
        generated_midi.writeFile(output_file)

    return 'q10.mid' 